# 01 — ZIT only (raw, 전처리 없음)

ZITboost 단독 HPO + refit + 후처리. **전처리를 일절 하지 않고** HP만 탐색한다.

- **PP**: Stage 0(웨이퍼맵 수동 제외)만 적용 — cleaning / imputation / outlier winsorize / 상관 제거 전부 SKIP
- **이유**: 전처리가 제거한 feature와 채워 넣은 imputation 값이 없는 *다른 feature space*에서 학습 → 스태킹 다양성 확보
- **LightGBM NaN 처리**: native sparse split으로 NaN을 별도 처리, imputation 불필요
- **출력**: `4_output/01_zit/zit_only_raw/{best_params.json, fold_models.pkl, optuna_*.db, oof|val|test_die.csv, oof|val|test_unit.csv}`

## 1. 환경 설정 + import

In [ ]:
import os, sys

# Google Drive 파일 ID들 — Colab에서 코드/데이터/모듈 zip을 자동으로 받아 풀 때 사용 (로컬은 무시)
GDRIVE_CODE_ID          = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'  # code.zip = setup.py + utils/
GDRIVE_DATASET_ID       = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'  # dataset.zip = 원본 CSV 4개
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'  # preprocessing.zip (raw는 EXCLUDE_COLS만 사용)
GDRIVE_MODELING_ID      = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'  # modeling.zip = 3_modeling/modules (코드 수정 시 재업로드)
GDRIVE_OUTPUT_ID        = '1ts73qEMmjX8cKIb-QeDQ-TMeyudFGWzs'  # 4_output.zip = 기존 실험 산출물 (RESUME 시 복원용)
RESUME                  = True  # True=기존 optuna db에 trial 이어 붙임 / False=처음부터 (db 있으면 의도적 에러)

try:
    import google.colab
    # Colab: 프로젝트 코드가 없으면 Drive에서 받아 /content/project에 설치
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    # 전처리 모듈: raw 노트북에서도 EXCLUDE_COLS 상수 가져오기 위해 필요
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    # 모델링 모듈: zit.py, hpo.py, postprocess.py 등 포함
    if GDRIVE_MODELING_ID and not os.path.exists('/content/project/3_modeling/modules/zit.py'):
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modeling.zip')
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system('unzip -qo /content/modeling.zip -d /content/project/3_modeling')
    # RESUME이면 이전 4_output을 통째로 복원 (이미 폴더 있으면 skip)
    if RESUME and GDRIVE_OUTPUT_ID and not os.path.exists('/content/project/4_output/01_zit'):
        os.system(f'gdown {GDRIVE_OUTPUT_ID} -O /content/4_output.zip')
        os.system('unzip -qo /content/4_output.zip -d /content/project')
        os.remove('/content/4_output.zip')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    # 로컬: setup.py가 두 단계 위에 있음
    %run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# 전처리 모듈 경로 추가 — EXCLUDE_COLS 상수(Stage 0 웨이퍼맵 제외 목록) 접근용
PP_DIR = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PP_DIR not in sys.path:
    sys.path.insert(0, PP_DIR)

# 모델링 모듈 경로 추가 — hpo, postprocess, zit, meta_features 임포트용
MOD_DIR = os.path.join(PROJECT_ROOT, '3_modeling')
if MOD_DIR not in sys.path:
    sys.path.insert(0, MOD_DIR)

# preprocess.run()은 사용 안 함 — EXCLUDE_COLS(Stage 0 제외 목록)만 가져옴
from modules.preprocess import EXCLUDE_COLS as _WAFER_MAP_EXCLUDE
from modules import hpo, postprocess              # hpo.run_hpo/refit_best/save_artifacts, postprocess.tune_and_apply
from meta_features import add_meta_features       # die_xy / position 메타피처 헬퍼

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.model_selection import KFold

import logging, time
logging.getLogger('lightgbm').setLevel(logging.ERROR)   # LGBM 상세 로그 억제
optuna.logging.set_verbosity(optuna.logging.WARNING)    # Optuna 경고만 표시

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'optuna v{optuna.__version__}')

# ZITboostRegressor: ZI-Tweedie + LightGBM EM 알고리즘 구현체 (Gu 2024)
from modules.zit import ZITboostRegressor

## 2. 실험 설정

- `PP_FIXED` 없음 (전처리 안 함)
- `ZIT_RAW_ANCHOR`: 전처리 버전 best HP를 탐색 시작점으로 활용 (feature space가 달라 최적값은 달라질 수 있음)
- `ZIT_SEARCH`: 넓은 1차 탐색 범위 (raw 첫 실험이므로 ±30% 좁히기 미적용)

In [ ]:
EXP_ID = 'zit-only-raw-001'  # 실험 식별자 — optuna study_name + 산출물 폴더에 사용
USER   = 'jh'                # DB 파일명 접두어 (멀티유저 환경에서 study 충돌 방지)

N_TRIALS         = 1         # 외부 런스크립트 타임아웃이 실제 종료를 결정하므로 충분히 크게
N_STARTUP_TRIALS = 30        # TPE 학습 전 무작위 탐색 trial 수 (warm-up)
TIMEOUT_SEC      = 90 * 60 * 60  # 90h 안전망 — 노트북 단독 실행 시 보호장치
N_FOLDS          = 5         # OOF RMSE 안정성 vs 연산 비용 균형
N_JOBS           = 7         # 모델 학습 병렬도 (ZITboost 내부 LGBM 스레드 수)

# 산출물 경로 — 기존 비-raw 디렉토리(zit_only/)와 분리하여 feature space 혼동 방지
OUT_DIR = os.path.join(OUTPUT_DIR, '01_zit', 'zit_only', 'raw', EXP_ID.split('-')[-1])
os.makedirs(OUT_DIR, exist_ok=True)
# study DB — RESUME 시 이 파일에서 이어서 탐색, 없으면 신규 생성
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')

CLIP_Y_EXTREME = True  # train Y의 극단값(1.0 1건)을 두 번째로 큰 값으로 clip (이상치 학습 안정화)

# anchor: 전처리 버전 best HP를 raw 탐색 시작점으로 사용
# raw에서는 feature space가 달라 최적값이 달라질 수 있으므로 Optuna가 자동 재탐색
ZIT_RAW_ANCHOR = {
    'zeta':                  1.149,   # Tweedie power (1.0=Poisson, 2.0=Gamma 사이 — 건강데이터 특성)
    'n_em_iters':            13,      # EM 반복 수 — 많을수록 수렴 확실, 느려짐
    'mu_n_estimators':       240,     # mu(기댓값) LightGBM 트리 수
    'mu_learning_rate':      0.00309, # mu 학습률 — 낮을수록 안정적, 더 많은 트리 필요
    'mu_num_leaves':         212,     # mu 잎 수 — 복잡도 조절
    'mu_max_depth':          3,       # mu 트리 깊이 상한
    'mu_min_child_samples':  132,     # mu 잎 최소 샘플 수 (과적합 방지)
    'mu_subsample':          0.649,   # mu 행 샘플링 비율
    'mu_colsample_bytree':   0.255,   # mu 열 샘플링 비율
    'mu_reg_alpha':          0.00576, # mu L1 정규화
    'mu_reg_lambda':         0.00155, # mu L2 정규화
    'pi_n_estimators':       125,     # pi(영 확률) LightGBM 트리 수
    'pi_learning_rate':      0.0408,  # pi 학습률
    'pi_num_leaves':         165,     # pi 잎 수
    'pi_max_depth':          11,      # pi 트리 깊이 상한
    'pi_min_child_samples':  38,      # pi 잎 최소 샘플 수
    'phi_n_estimators':      57,      # phi(분산) LightGBM 트리 수
    'phi_learning_rate':     0.00628, # phi 학습률
    'phi_num_leaves':        65,      # phi 잎 수
    'phi_max_depth':         4,       # phi 트리 깊이 상한
    'phi_min_child_samples': 190,     # phi 잎 최소 샘플 수
}
ANCHOR_TAU_PI = 0.944  # pi > tau_pi 이면 예측을 0으로 처리하는 임계값

# 탐색 공간: raw 첫 탐색이므로 1차 pass 수준의 넓은 범위로 설정
ZIT_SEARCH = {
    'zeta':                  {'type': 'float', 'low': 1.05,   'high': 1.90,   'log': False},
    'n_em_iters':            {'type': 'int',   'low': 5,      'high': 20},
    'mu_n_estimators':       {'type': 'int',   'low': 50,     'high': 500},
    'mu_learning_rate':      {'type': 'float', 'low': 0.001,  'high': 0.05,   'log': True},
    'mu_num_leaves':         {'type': 'int',   'low': 31,     'high': 300},
    'mu_max_depth':          {'type': 'int',   'low': 3,      'high': 8},
    'mu_min_child_samples':  {'type': 'int',   'low': 20,     'high': 300},
    'mu_subsample':          {'type': 'float', 'low': 0.40,   'high': 1.0,    'log': False},
    'mu_colsample_bytree':   {'type': 'float', 'low': 0.10,   'high': 0.60,   'log': False},
    'mu_reg_alpha':          {'type': 'float', 'low': 1e-5,   'high': 1.0,    'log': True},
    'mu_reg_lambda':         {'type': 'float', 'low': 1e-5,   'high': 1.0,    'log': True},
    'pi_n_estimators':       {'type': 'int',   'low': 50,     'high': 400},
    'pi_learning_rate':      {'type': 'float', 'low': 0.005,  'high': 0.10,   'log': True},
    'pi_num_leaves':         {'type': 'int',   'low': 31,     'high': 300},
    'pi_max_depth':          {'type': 'int',   'low': 5,      'high': 16},
    'pi_min_child_samples':  {'type': 'int',   'low': 10,     'high': 100},
    'phi_n_estimators':      {'type': 'int',   'low': 20,     'high': 200},
    'phi_learning_rate':     {'type': 'float', 'low': 0.002,  'high': 0.050,  'log': True},
    'phi_num_leaves':        {'type': 'int',   'low': 20,     'high': 200},
    'phi_max_depth':         {'type': 'int',   'low': 3,      'high': 8},
    'phi_min_child_samples': {'type': 'int',   'low': 30,     'high': 300},
}
TAU_PI_RANGE = (0.5, 1.0)  # tau_pi도 HP로 탐색 — pi 모델 품질에 따라 최적값이 크게 달라짐

print(f'EXP_ID={EXP_ID} | USER={USER} | raw_mode=True')
print(f'N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS} | N_JOBS={N_JOBS}')
print(f'OUT_DIR={OUT_DIR}')
print(f'ZIT search HP={len(ZIT_SEARCH)} + tau_pi=1 = {len(ZIT_SEARCH)+1}')

## 3. 데이터 로드 (raw — Stage 0만 적용)

`load_all()`이 이미 처리하는 것:
- 전체 행 결측(all-NaN) 407개 제거
- position 4개 미만 unit 1개 제거

이 셀에서 추가로 하는 것:
- Stage 0(웨이퍼맵 수동 제외) 적용
- 그 외 cleaning / imputation / outlier 일절 없음

In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

# train y의 극단값(1.0 1건)만 두 번째로 큰 값으로 clip (학습 입력 안정화)
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 -> {second_max:.6f} clip, {n_clipped}개 샘플')

# Stage 0만 적용: 웨이퍼맵 수동 제외 리스트(_WAFER_MAP_EXCLUDE)로 feature 사전 제거
# cleaning / imputation / outlier winsorize 는 일절 하지 않음
# LightGBM은 NaN을 네이티브로 처리하므로 imputation이 불필요
n_before = len(feat_cols)
feat_cols_raw = [c for c in feat_cols if c not in _WAFER_MAP_EXCLUDE]
print(f'[Stage 0] 웨이퍼맵 사전 제외: {n_before} -> {len(feat_cols_raw)} ({n_before - len(feat_cols_raw)}개 제거)')
print('[raw mode] cleaning / imputation / outlier winsorize / 상관 제거 전부 SKIP')
print('  LightGBM NaN 네이티브 처리, 원본 feature 분포 그대로 유지')

xs_train = xs_dict['train'].copy()
xs_val   = xs_dict['validation'].copy()
xs_test  = xs_dict['test'].copy()

# 메타피처 추가 (position raw 정수 + die_x/die_y 연속형)
feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_raw,
    position_mode='raw', use_die_xy=True,
)

# feature 행렬 numpy float64 (NaN 포함 그대로)
X_train = xs_train[feat_cols_clean].values.astype(np.float64)
X_val   = xs_val[feat_cols_clean].values.astype(np.float64)
X_test  = xs_test[feat_cols_clean].values.astype(np.float64)

uid_train_die = xs_train[KEY_COL].values
uid_val_die   = xs_val[KEY_COL].values
uid_test_die  = xs_test[KEY_COL].values

y_train_unit_s = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit_s   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit_s  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

y_train_die = xs_train[KEY_COL].map(y_train_unit_s).values.astype(np.float64)

print(f'[데이터 준비] feat_cols_clean: {len(feat_cols_clean)}')
print(f'  X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')
print(f'  unit train={len(y_train_unit_s):,}, val={len(y_val_unit_s):,}, test={len(y_test_unit_s):,}')
nan_pct = np.isnan(X_train).mean() * 100
print(f'  NaN in X_train: {np.isnan(X_train).sum():,} ({nan_pct:.1f}%) — LightGBM이 직접 처리')


## 4. K-fold split + Optuna objective

- KFold: unit ID 단위 분할 (같은 unit의 4 die는 같은 fold)
- τ_π: HP로 탐색 (0.5~1.0)
- die-level π/μ → τ_π 적용 → unit 평균 → OOF unit RMSE

In [ ]:
# unit ID 기준 KFold — 같은 unit의 4 die가 train/val에 섞이면 leakage 발생하므로 unit 단위 분할 필수
unique_units = y_train_unit_s.index.values
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf.split(unique_units))


def _apply_tau_pi(pred_die, pi_die, tau_pi):
    # pi > tau_pi: 구조적 0 영역으로 판정 → 예측을 0으로 강제
    return np.where(pi_die > tau_pi, 0.0, pred_die)


def _mean_die_to_unit(pred_die, uid_die):
    # die 4개의 예측을 unit 단위로 평균 집계 → unit RMSE 계산 준비
    df = pd.DataFrame({KEY_COL: uid_die, 'pred': pred_die})
    return df.groupby(KEY_COL, sort=False)['pred'].mean().reset_index()


def objective(trial):
    t0 = time.time()
    # ZIT_SEARCH 딕셔너리에서 trial 별 HP 샘플링 (hpo.sample_from_space 내부에서 type 분기)
    params = hpo.sample_from_space(trial, ZIT_SEARCH)
    # tau_pi는 ZIT 모델 파라미터가 아닌 후처리 임계값 — HP로 같이 탐색
    tau_pi = trial.suggest_float('tau_pi', TAU_PI_RANGE[0], TAU_PI_RANGE[1])

    # ZITboostRegressor 고정 파라미터 (탐색 대상 아님)
    params['random_state'] = SEED   # 재현성
    params['n_jobs']       = N_JOBS # LightGBM 내부 스레드 수
    params['verbose']      = -1     # LightGBM 출력 억제
    params['device']       = 'cpu'  # GPU 미사용 (EM 루프 안에서 반복 호출이 많아 전환 오버헤드 큼)
    params['em_tol']       = 1e-7   # EM 수렴 판정 허용 오차

    fold_oof_rmse = []
    # OOF 예측 버퍼 — fold별로 채워나가, 전체 train RMSE 계산에 사용
    oof_pred_unit = pd.Series(np.nan, index=y_train_unit_s.index, dtype=np.float64)

    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tr_units = unique_units[tr_uidx]
        vl_units = unique_units[vl_uidx]
        # unit mask → die mask로 변환 (unit 단위 분할을 die 행에 적용)
        tr_mask = np.isin(uid_train_die, tr_units)
        vl_mask = np.isin(uid_train_die, vl_units)

        model = ZITboostRegressor(**params)
        model.fit(X_train[tr_mask], y_train_die[tr_mask])

        # predict_components: π(zero_prob), μ(mean), φ(dispersion) 세 성분 반환
        pi_vl, mu_vl, _ = model.predict_components(X_train[vl_mask])
        pred_die_raw   = np.clip((1 - pi_vl) * mu_vl, 0, None)  # E[Y] = (1-π)·μ, 음수 방지 clip
        pred_die_taupi = _apply_tau_pi(pred_die_raw, pi_vl, tau_pi)
        unit_pred_df   = _mean_die_to_unit(pred_die_taupi, uid_train_die[vl_mask])

        oof_pred_unit.loc[unit_pred_df[KEY_COL].values] = unit_pred_df['pred'].values
        y_vl = y_train_unit_s.loc[unit_pred_df[KEY_COL].values].values
        fold_rmse = float(np.sqrt(np.mean((unit_pred_df['pred'].values - y_vl) ** 2)))
        fold_oof_rmse.append(fold_rmse)

        # MedianPruner: 중간 fold까지의 평균 RMSE 보고 → 나쁘면 조기 종료
        avg = float(np.mean(fold_oof_rmse))
        trial.report(avg, step=fold_idx)
        if trial.should_prune():
            trial.set_user_attr('pruned_at_fold', fold_idx + 1)
            trial.set_user_attr('elapsed_sec', time.time() - t0)
            trial.set_user_attr('tau_pi', tau_pi)
            raise optuna.TrialPruned()

    if oof_pred_unit.isna().any():
        raise RuntimeError('OOF NaN — fold 누락')

    # 전체 OOF unit RMSE — Optuna 최소화 목표
    oof_rmse = float(np.sqrt(np.mean((oof_pred_unit.values - y_train_unit_s.values) ** 2)))
    elapsed  = time.time() - t0
    # user_attr: study DB에 영구 저장 — 후분석 및 best trial 재현용
    trial.set_user_attr('elapsed_sec', elapsed)
    trial.set_user_attr('tau_pi', tau_pi)
    trial.set_user_attr('fold_oof_rmse', fold_oof_rmse)
    print(f'  trial #{trial.number}: tau_pi={tau_pi:.3f}, oof={oof_rmse:.6f}, elapsed={elapsed:.0f}s')
    return oof_rmse


print(f'fold split: {N_FOLDS} folds, unit 단위 분할, seed={SEED}')

## 5. Optuna study + anchor enqueue + optimize

In [ ]:
# TPE: multivariate=HP 결합 분포 학습, group=조건부 축 자동 skip, seed=None → run마다 다양성
sampler = TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS)
# MedianPruner: n_startup_trials 이후부터 중앙값 대비 나쁜 trial 가지치기 (n_warmup_steps=2: fold 2개 후 판정)
pruner  = MedianPruner(n_startup_trials=N_STARTUP_TRIALS, n_warmup_steps=2)

# load_if_exists=RESUME: RESUME=True이면 기존 DB 이어서, False이면 신규 (DB 있으면 DuplicatedStudyError)
study = optuna.create_study(
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    sampler=sampler, pruner=pruner,
    direction='minimize',           # OOF RMSE 최소화
    load_if_exists=RESUME,
)

# anchor enqueue: trial 0에 전처리 버전 best HP 주입 → raw 탐색의 warm-start
ANCHOR_FOR_ENQUEUE = dict(ZIT_RAW_ANCHOR)
ANCHOR_FOR_ENQUEUE['tau_pi'] = ANCHOR_TAU_PI
if len(study.trials) == 0:
    hpo.enqueue_anchor(study, ANCHOR_FOR_ENQUEUE)
else:
    # RESUME 시 재enqueue 금지 — anchor를 다시 넣으면 trial 0을 덮어쓰며 best 오염 가능
    print(f'[enqueue skip] 기존 trial {len(study.trials)} 있음 — resume')

# study_meta: study DB에 박제할 재현성 메타 — 어떤 조건으로 학습됐는지 DB만 보고도 알 수 있게
study_meta = {
    'exp_id': EXP_ID, 'user': USER, 'model': 'ZITboost (zit_only_raw)',
    'raw_mode': 'True',                        # raw임을 명시 (분석/필터링용 마커)
    'n_trials': N_TRIALS, 'n_folds': N_FOLDS, 'n_jobs': N_JOBS,
    'pp_fixed': '{}',                          # raw mode — preprocess.run 미사용 (빈 dict로 박제)
    'anchor': ZIT_RAW_ANCHOR, 'anchor_tau_pi': ANCHOR_TAU_PI,
    'sampler': 'TPE seed=None multivariate group',
    'pruner':  f'MedianPruner n_startup={N_STARTUP_TRIALS} n_warmup=2',
    'CLIP_Y_EXTREME': CLIP_Y_EXTREME, 'SEED': int(SEED),
}
for k, v in study_meta.items():
    study.set_user_attr(k, str(v))

print(f'study: {study.study_name}  DB: {DB_PATH}')
print(f'기존 trial: {len(study.trials)}')

# n_jobs=1: objective 내부에서 이미 N_JOBS 병렬화 → 외부 병렬화 시 메모리 폭발 위험
t_start = time.time()
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, n_jobs=1, show_progress_bar=True)
print(f'\n[HPO 완료] 전체 {time.time()-t_start:.0f}s, total trials={len(study.trials)}')
print(f'  best OOF RMSE: {study.best_value:.6f}')

## 6. Best trial 정보 + anchor enqueue 검증

In [ ]:
best_trial = study.best_trial
best_params = best_trial.params
best_tau_pi = float(best_trial.user_attrs.get('tau_pi', ANCHOR_TAU_PI))

print(f'=== Best Trial #{best_trial.number} ===')
print(f'  OOF RMSE  : {best_trial.value:.6f}')
print(f'  best tau_pi : {best_tau_pi:.4f}')
print(f'  elapsed   : {best_trial.user_attrs.get("elapsed_sec", 0):.0f}s')
for k, v in sorted(best_params.items()):
    print(f'    {k}: {v}')

trial0 = study.trials[0]
anchor_check = all(
    abs(float(trial0.params.get(k, float('nan'))) - float(v)) < 1e-9
    if not isinstance(v, str) else trial0.params.get(k) == v
    for k, v in ANCHOR_FOR_ENQUEUE.items()
)
print(f'\n[검증] trial 0 == anchor? {anchor_check}')


## 7. Best HP 5-fold refit + die-level π/μ 캡처

In [ ]:
from modules.zit import ZITboostRegressor

# tau_pi는 후처리 파라미터 — model.fit에 넘기지 않고 별도 보관
best_full_params = {k: v for k, v in best_params.items() if k != 'tau_pi'}
# 고정 파라미터 주입 (HPO 목적함수와 동일하게 설정)
best_full_params['random_state'] = SEED
best_full_params['n_jobs']       = N_JOBS
best_full_params['verbose']      = -1
best_full_params['device']       = 'cpu'
best_full_params['em_tol']       = 1e-7

n_train_die = len(X_train)
n_val_die   = len(X_val)
n_test_die  = len(X_test)

# OOF: fold별로 채워나가는 배열 — 전체 train 예측 재구성에 사용
oof_die_pi   = np.full(n_train_die, np.nan)
oof_die_mu   = np.full(n_train_die, np.nan)
oof_die_pred = np.full(n_train_die, np.nan)
# val/test: fold별 예측을 평균 — 앙상블 효과로 단일 fold보다 안정적
val_die_pi   = np.zeros(n_val_die)
val_die_mu   = np.zeros(n_val_die)
val_die_pred = np.zeros(n_val_die)
test_die_pi  = np.zeros(n_test_die)
test_die_mu  = np.zeros(n_test_die)
test_die_pred = np.zeros(n_test_die)

fold_models          = []   # 나중에 pkl로 저장 (스태킹 메타피처 생성 등 재사용)
em_history_per_fold  = []   # fold별 EM 수렴 이력 — 수렴 여부 및 반복 수 검증용

print('=== Best HP 5-fold refit (ZITboost raw) ===')
t0 = time.time()
for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
    tr_units = unique_units[tr_uidx]
    vl_units = unique_units[vl_uidx]
    tr_mask  = np.isin(uid_train_die, tr_units)
    vl_mask  = np.isin(uid_train_die, vl_units)

    model = ZITboostRegressor(**best_full_params)
    model.fit(X_train[tr_mask], y_train_die[tr_mask])

    # OOF: val_mask 행에 π/μ/pred 채우기
    pi_vl, mu_vl, _ = model.predict_components(X_train[vl_mask])
    oof_die_pi[vl_mask]   = pi_vl
    oof_die_mu[vl_mask]   = mu_vl
    oof_die_pred[vl_mask] = np.clip((1 - pi_vl) * mu_vl, 0, None)

    # val/test: N_FOLDS개 모델 예측 평균 누적
    pi_v, mu_v, _ = model.predict_components(X_val)
    pi_t, mu_t, _ = model.predict_components(X_test)
    val_die_pi   += pi_v / N_FOLDS;  val_die_mu   += mu_v / N_FOLDS
    val_die_pred += np.clip((1 - pi_v) * mu_v, 0, None) / N_FOLDS
    test_die_pi  += pi_t / N_FOLDS;  test_die_mu  += mu_t / N_FOLDS
    test_die_pred += np.clip((1 - pi_t) * mu_t, 0, None) / N_FOLDS

    fold_models.append(model)
    em_history_per_fold.append(model.em_history_)
    print(f'  fold {fold_idx+1}/{N_FOLDS} done ({time.time()-t0:.0f}s)')

assert not np.isnan(oof_die_pred).any(), 'OOF die pred 미커버'

# EM 수렴 검증: 단조 감소(monotonic=True) + 마지막 RMSE 확인
print('\n[EM 수렴 체크]')
for f, hist in enumerate(em_history_per_fold):
    key = 'unit_rmse' if hist and 'unit_rmse' in hist[0] else 'rmse'
    rmses = [h[key] for h in hist]
    mono  = all(rmses[i+1] <= rmses[i] + 1e-6 for i in range(len(rmses)-1))
    print(f'  fold {f+1}: {len(hist)} EM iter, last_{key}={rmses[-1]:.6f}, monotonic={mono}')

print('\n[refit 완료]')

## 8. 후처리 — τ_π 적용 → 집계 8 + position Optuna + zero_clip(log)

In [ ]:
# tau_pi 적용: pi > best_tau_pi 인 die → 구조적 0 판정, 예측 강제 0
oof_die_pred_taupi  = _apply_tau_pi(oof_die_pred,  oof_die_pi,  best_tau_pi)
val_die_pred_taupi  = _apply_tau_pi(val_die_pred,  val_die_pi,  best_tau_pi)
test_die_pred_taupi = _apply_tau_pi(test_die_pred, test_die_pi, best_tau_pi)

# tau_pi가 0으로 처리한 die 비율 — 너무 높으면 pi 모델이 과도하게 aggressive함을 의미
killed = {
    'oof':  float((oof_die_pi  > best_tau_pi).mean()),
    'val':  float((val_die_pi  > best_tau_pi).mean()),
    'test': float((test_die_pi > best_tau_pi).mean()),
}
print(f'[tau_pi={best_tau_pi:.3f} 적용] 0 처리 die 비율: {killed}')

# postprocess.tune_and_apply: 집계방식(8종) × position가중치 × zero_clip 을 Optuna로 동시 최적화
pp_res = postprocess.tune_and_apply(
    xs_train, xs_val, xs_test,
    die_pred_train=oof_die_pred_taupi,
    die_pred_val=val_die_pred_taupi,
    die_pred_test=test_die_pred_taupi,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    use_pi_threshold=False,          # tau_pi 적용은 이미 위에서 완료 — postprocess 내 중복 적용 방지
    agg_methods=postprocess.AGG_METHODS,  # mean, max, min, median, std, q25, q75, range
    zero_clip_log_space=True,        # log 스케일로 zero_clip 탐색 → 미세 조정에 효과적
    position_method='optuna',        # 4개 position 가중치를 Optuna로 탐색
    position_optuna_n_trials=50,     # postprocess 내 Optuna trial 수 (빠른 수렴 목적)
)

print(f'\n[Postprocess]')
print(f'  best_agg            : {pp_res["best_agg"]}')
print(f'  pos_weights         : {pp_res["pos_weights"]}')
print(f'  best_zero_clip(log) : {pp_res["best_zero_clip"]}')
print(f'  train_rmse          : {pp_res["train_rmse"]:.6f}')

# val/test RMSE는 pp_res에 포함될 수도, 없을 수도 있으므로 None 가드
if pp_res.get('final_val_unit') is not None:
    _val_pred = pp_res['final_val_unit'].set_index(KEY_COL)['pred'].loc[y_val_unit_s.index]
    val_rmse  = float(np.sqrt(np.mean((_val_pred.values  - y_val_unit_s.values)  ** 2)))
    print(f'  val_rmse            : {val_rmse:.6f}')
if pp_res.get('final_test_unit') is not None:
    _test_pred = pp_res['final_test_unit'].set_index(KEY_COL)['pred'].loc[y_test_unit_s.index]
    test_rmse  = float(np.sqrt(np.mean((_test_pred.values - y_test_unit_s.values) ** 2)))
    print(f'  test_rmse           : {test_rmse:.6f}')

print(f'  agg_rmses           : {pp_res["agg_rmses"]}')
for k, v in pp_res.get('decisions', {}).items():
    print(f'    {k:14s} {v}')

## 9. 산출물 9개 저장

In [ ]:
import json, pickle, hashlib

# fold_models.pkl: 5개 모델 객체 + 메타 → 스태킹용 메타피처 재생성에 재사용 가능
with open(os.path.join(OUT_DIR, 'fold_models.pkl'), 'wb') as f:
    pickle.dump({
        'fold_models':         fold_models,
        'feature_names':       feat_cols_clean,
        'model_name':          'zitboost_raw',
        'n_folds':             N_FOLDS,
        'em_history_per_fold': em_history_per_fold,
    }, f)

# unit_ids_hash: 학습 데이터 구성 검증용 — 재현 시 동일 hash 확인
uid_arr = ys_input['train'][KEY_COL].unique()
unit_ids_hash = hashlib.sha1(','.join(map(str, uid_arr)).encode()).hexdigest()

# best_params.json: 실험 재현에 필요한 모든 정보를 한 파일에 박제
best_meta = {
    'exp_id':                EXP_ID,
    'model_name':            'zitboost_raw',
    'raw_mode':              True,                      # 전처리 없음 명시
    'best_trial_number':     best_trial.number,
    'best_oof_rmse':         float(best_trial.value),
    'best_params_resolved':  best_full_params,
    'best_tau_pi':           best_tau_pi,
    'feature_names':         feat_cols_clean,
    'n_features':            len(feat_cols_clean),
    'n_folds':               N_FOLDS,
    'unit_ids_hash':         unit_ids_hash,
    'n_units_train':         int(len(uid_arr)),
    'effective_pp_params':   {},                        # raw mode — 빈 dict
    'study_meta':            study_meta,
    'postprocess': {
        'best_agg':            pp_res['best_agg'],
        'pos_weights':         pp_res['pos_weights'].tolist() if pp_res['pos_weights'] is not None else None,
        'best_zero_clip':      float(pp_res['best_zero_clip']),
        'zero_clip_log_space': pp_res['zero_clip_log_space'],
        'position_method':     pp_res['position_method'],
        'agg_rmses':           {k: float(v) for k, v in pp_res['agg_rmses'].items()},
        'train_rmse':          float(pp_res['train_rmse']),
    },
}
with open(os.path.join(OUT_DIR, 'best_params.json'), 'w', encoding='utf-8') as f:
    json.dump(best_meta, f, indent=2, ensure_ascii=False, default=str)


def _build_die_df(uid, die_id, position, pi, mu, pred, y_unit):
    # die 수준 예측 DataFrame 구성 — π/μ/pred 성분을 함께 저장 (후분석, 앙상블 활용용)
    df = pd.DataFrame({
        KEY_COL: uid, DIE_KEY_COL: die_id, 'position': position,
        'pi': pi, 'one_minus_pi': 1.0 - pi, 'mu': mu, 'pred': pred,
    })
    if y_unit is not None:
        df[TARGET_COL] = df[KEY_COL].map(y_unit)
    return df

# die-level CSV 3개: oof(train fold OOF), val, test
_build_die_df(uid_train_die, xs_train[DIE_KEY_COL].values, xs_train['position'].values,
              oof_die_pi, oof_die_mu, oof_die_pred, y_train_unit_s
).to_csv(os.path.join(OUT_DIR, 'oof_die.csv'), index=False)
_build_die_df(uid_val_die, xs_val[DIE_KEY_COL].values, xs_val['position'].values,
              val_die_pi, val_die_mu, val_die_pred, y_val_unit_s
).to_csv(os.path.join(OUT_DIR, 'val_die.csv'), index=False)
_build_die_df(uid_test_die, xs_test[DIE_KEY_COL].values, xs_test['position'].values,
              test_die_pi, test_die_mu, test_die_pred, y_test_unit_s
).to_csv(os.path.join(OUT_DIR, 'test_die.csv'), index=False)


def _build_unit_df(unit_pred_df, y_unit):
    # unit 수준 예측 DataFrame — 실제 health 값 병합 (val/test는 비공개이므로 NaN이 될 수 있음)
    out = unit_pred_df.copy()
    out[TARGET_COL] = out[KEY_COL].map(y_unit)
    return out

# unit-level CSV 3개: 스태킹 meta-feature 및 최종 제출용
_build_unit_df(pp_res['final_train_unit'], y_train_unit_s).to_csv(os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(pp_res['final_val_unit'],   y_val_unit_s  ).to_csv(os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(pp_res['final_test_unit'],  y_test_unit_s ).to_csv(os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

print(f'\n저장 완료: {OUT_DIR}')
for fn in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, fn)) / 1024
    print(f'  {fn:30s}  {sz:10,.1f} KB')

# Colab에서만 실행: 산출물 zip 생성 후 자동 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip = shutil.make_archive(os.path.join('/content', 'zit_only_raw_001_outputs'), 'zip', OUT_DIR)
    print(f'\n[zip 생성] {_zip} ({os.path.getsize(_zip)/1024:.1f} KB)')
    try:
        files.download(_zip)
    except Exception as _e:
        # files.download가 실패하면 클릭 가능한 링크로 대체
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip))
except ImportError:
    pass